In [1]:
import numpy as np
import trimesh
import igl
from pyfaidx import Fasta
import meshplot as mp
import trimesh
import sys
import os 
import json
from PIL import Image, ImageDraw, ImageFont

file = open('../global/colors_500.txt', 'r')
lines = file.readlines()
colors = np.zeros((len(lines),3))
for i in range(len(lines)):
    colors[i] = np.array(lines[i].split())

In [2]:

def read_fidx(fidx_file):
    index_dict = []
    max_len = 0
    with open(fidx_file, 'r') as f:
        for line in f:
            index_dict.append(int(line.strip().split('\t')[0]))
            parts = line.strip().split('\t')
            max_len = max(len(parts), max_len)
            # print(parts)
            # if len(parts) < 5:
            #     continue
            
            # seq_name = parts[0]
            # seq_length = int(parts[1])
            # byte_offset = int(parts[2])
            # bases_per_line = int(parts[3])
            # bytes_per_line = int(parts[4])
            
            # index_dict[seq_name] = {
            #     seq_length,
            #     byte_offset,
            #     bases_per_line,
            #     bytes_per_line
            # }
    return index_dict

def show_segment(v, f, seg_index):
    f_color = np.zeros((f.shape[0], 3))
    for i in range(len(seg_index)):
        f_color[i] = colors[seg_index[i] % len(colors)]
    mp.plot(v, f, f_color, shading={'wireframe': False})

def read_seg(seg_file):
    seg_list = []
    with open(seg_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')[0]
            seg_list.append(int(parts))
    return seg_list


def generate_single_raw_ply(v, f, output_path, obj_name):
    mesh = trimesh.Trimesh(v, f)
    ply_path = os.path.join(output_path, obj_name+'.ply')
    mesh.export(ply_path)
    
    
def generate_single_color_ply(v, f, output_path, obj_name, seg_index):
    mesh = trimesh.Trimesh(v, f)
    
    f_color = np.zeros((f.shape[0], 3))
    for i in range(len(seg_index)):
        f_color[i] = colors[seg_index[i] % len(colors)]
    mesh.visual.face_colors = (f_color * 255).astype(np.uint8)
    
    ply_path = os.path.join(output_path, obj_name+'.ply')
    mesh.export(ply_path)

def generate_ply(obj_path, raw_output_path, seg_output_path):
    obj_names = []
    for file in os.listdir(obj_path):
        if file.endswith('.obj'):
            obj_names.append(file.split('/')[-1].split('.')[0])
    print(len(obj_names))
    
    for i in range(len(obj_names)):
        if (i % 1000) == 0:
            print(f'reach {i}-th object')
        obj_name = obj_names[i]

        obj_file = os.path.join(obj_path, obj_name+'.obj')
        v, f = igl.read_triangle_mesh(obj_file)
        
        seg_file = os.path.join(obj_path, obj_name+'.seg')
        seg_index = read_seg(seg_file)
        
        generate_single_raw_ply(v, f, raw_output_path, obj_name)
        generate_single_color_ply(v, f, seg_output_path, obj_name, seg_index)



seg_total_path = 'D:\\dataset\\fusion360gallery\\s2.0.1'
obj_path = 'D:\\dataset\\fusion360gallery\\s2.0.1\\meshes'
raw_output_path = 'D:\\dataset\\fusion360gallery\\s2.0.1\\ply_raw'
seg_output_path = 'D:\\dataset\\fusion360gallery\\s2.0.1\\ply_seg'
segment_names_file = 'D:\\dataset\\fusion360gallery\\s2.0.1\\segment_names.json'

# generate_ply(obj_path, raw_output_path, seg_output_path)

In [ ]:
def write_obj_names(obj_path, seg_total_path):
    obj_names = []
    for file in os.listdir(obj_path):
        if file.endswith('.obj'):
            obj_names.append(file.split('/')[-1].split('.')[0])
            
    with open(os.path.join(seg_total_path, 'obj_names.txt'), 'w') as f:
        for obj_name in obj_names:
            f.write(obj_name + '\n')
# write_obj_names(obj_path, seg_total_path)


In [ ]:
# read json file from segment_names_file
segment_names_f = open(segment_names_file, 'r')
segment_names = json.load(segment_names_f)

def generate_type_colormap():
    image = Image.new('RGB', (800, 100), 'white')
    draw = ImageDraw.Draw(image)

    labels = [
        "ExtrudeSide",
        "ExtrudeEnd",
        "CutSide",
        "CutEnd",
        "Fillet",
        "Chamfer",
        "RevolveSide",
        "RevolveEnd"
    ]

    block_width = 100
    block_height = 100

    
    # font = ImageFont.load_default()
    font = ImageFont.truetype("arial.ttf", 16)
    for i in range(8):
        row = i // 8
        col = i % 8
        x0 = col * block_width
        y0 = row * block_height
        x1 = x0 + block_width
        y1 = y0 + block_height
        
        cur_color = colors[i] * 255
        cur_color = tuple([int(c) for c in cur_color])
        
        # Draw colored rectangle
        draw.rectangle([x0, y0, x1, y1], fill=cur_color)
        
        # Draw border
        draw.rectangle([x0, y0, x1, y1], outline='black', width=2)
        
        # Calculate text position (centered)
        text = labels[i]
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        text_x = x0 + (block_width - text_width) // 2
        text_y = y0 + (block_height - text_height) // 2
        
        # Draw text (black or white depending on background brightness)
        text_color = 'black'
        draw.text((text_x, text_y), text, fill=text_color, font=font)

    # Save the image
    image.save('fusion360_colormap.png')
    # image.show()
    
# generate_type_colormap()

(0, 255, 0)
(255, 0, 255)
(0, 127, 255)
(255, 127, 0)
(127, 191, 127)
(94, 2, 160)
(223, 0, 10)
(214, 253, 0)
